# Complete engineering workflow — tools only, every step explicit

CAD → inspect → FEA → change the design → FEA again → print → CFD. The tools talk through files
(STEP, STL). Nothing runs unless you run the cell; you can stop after any step.

In [ ]:
import math
from pathlib import Path
from vegeta import dedalus, talos, aeromant, mellonia
from vegeta.dedalus.examples import Bracket, StreamlinedBody
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM

RUNS = Path("_runs/workflow"); RUNS.mkdir(parents=True, exist_ok=True)

## 1. CAD: bracket, 6 mm thick

In [ ]:
g6 = Bracket().generate(thickness=6.0)
cad6 = g6.export(RUNS / "bracket_t6")
g6

## 2. Inspect the STEP to choose the clamped and loaded faces

In [ ]:
talos.inspect_step(cad6.artifacts["step"], units="mm-N-MPa")

## 3. FEA: clamp x = -40, 200 N down on x = +40 (aluminium, explicit values)

In [ ]:
aluminium = talos.Material("Al 6061-T6", youngs_modulus=68900, poissons_ratio=0.33, yield_strength=276,
                           source="nominal handbook values")

def bracket_model(step):
    return talos.StructuralModel(
        geometry=step, units="mm-N-MPa", material=aluminium,
        regions=[talos.SurfacesOnPlane("clamped", "x", -40.0), talos.SurfacesOnPlane("loaded", "x", 40.0)],
        supports=[talos.FixedSupport("clamped")], loads=[talos.Force("loaded", fz=-200.0)],
        mesh_settings=talos.MeshSettings(element_size=3.0))

m6 = bracket_model(cad6.artifacts["step"])
m6.mesh(RUNS / "fea_t6")
fea6 = m6.solve(RUNS / "fea_t6")
fea6

## 4. The engineer decides: make it thicker, generate again, run FEA again

In [ ]:
g8 = Bracket().generate(thickness=8.0)
cad8 = g8.export(RUNS / "bracket_t8")
m8 = bracket_model(cad8.artifacts["step"])
m8.mesh(RUNS / "fea_t8")
fea8 = m8.solve(RUNS / "fea_t8")
for name, r in (("6 mm", fea6), ("8 mm", fea8)):
    print(f"{name}: max deflection {r.metrics['max_displacement']:.3f} mm, "
          f"max von Mises {r.metrics['max_von_mises']:.1f} MPa, SF {r.metrics['safety_factor_yield']:.2f}")

## 5. Manufacturability of the chosen bracket (orientation chosen by the engineer)

In [ ]:
prn8 = mellonia.slice_stl(cad8.artifacts["stl"], GENERIC_PLA_0_2MM, mellonia.Orientation(), RUNS / "print_t8")
prn8.metrics["estimated_time"], prn8.metrics["filament_used_g"]

## 6. CFD on a different part: a streamlined body in air at 10 m/s
Coarse RANS template (trend-level results; see the template notes). This cell takes about 30 s.

In [ ]:
body = StreamlinedBody().generate()
cad_body = body.export(RUNS / "body", formats=("stl",), stl_tolerance=0.05)
case = aeromant.CFDCase(
    "rans_ksst_external_simplefoam", cad_body.artifacts["stl"],
    dict(velocity=10.0, kinematic_viscosity=1.5e-5, density=1.2, reference_area=math.pi * 0.01**2,
         reference_length=0.1, center_of_rotation=(0.05, 0, 0), iterations=300),
    workdir=RUNS / "body_cfd", geometry_units="mm", environment=aeromant.OpenFOAMEnvironment.detect())
case.prepare(overwrite=True)
cfd = case.run(progress=True)
cfd.metrics["Cd"], cfd.metrics["converged"]

## What was produced
Every step left its native files next to a `summary.json` under `_runs/workflow/`. Keeping track of
which geometry each result belongs to is manual here — that is what `vegeta.core` revisions do
(see `06_core_revisions.ipynb`).

In [ ]:
for p in sorted(RUNS.iterdir()):
    print(p.name, "->", sorted(x.name for x in p.iterdir())[:6])